# Agentic Workflow: Simple Query-Response System

This notebook demonstrates a simple agentic workflow that:
- Maintains a pre-defined knowledge dataset
- Processes user queries intelligently
- Fetches and returns relevant information
- Handles various query patterns

## 1. Import Required Libraries

In [ ]:
import json
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from datetime import datetime

## 2. Define the Pre-defined Dataset

Our dataset contains information about products in an e-commerce system.

In [ ]:
# Pre-defined dataset: Product Information
PRODUCT_DATABASE = [
    {
        "id": "P001",
        "name": "Laptop Pro 15",
        "category": "Electronics",
        "price": 1299.99,
        "stock": 45,
        "description": "High-performance laptop with 16GB RAM and 512GB SSD",
        "specs": {
            "processor": "Intel i7",
            "ram": "16GB",
            "storage": "512GB SSD"
        }
    },
    {
        "id": "P002",
        "name": "Wireless Mouse",
        "category": "Accessories",
        "price": 29.99,
        "stock": 150,
        "description": "Ergonomic wireless mouse with 6 programmable buttons",
        "specs": {
            "connectivity": "Bluetooth 5.0",
            "battery_life": "6 months"
        }
    },
    {
        "id": "P003",
        "name": "USB-C Hub",
        "category": "Accessories",
        "price": 49.99,
        "stock": 0,
        "description": "7-in-1 USB-C hub with HDMI, USB 3.0, and SD card reader",
        "specs": {
            "ports": "HDMI, 3x USB 3.0, SD/microSD, USB-C PD"
        }
    },
    {
        "id": "P004",
        "name": "Mechanical Keyboard",
        "category": "Accessories",
        "price": 89.99,
        "stock": 30,
        "description": "RGB backlit mechanical keyboard with blue switches",
        "specs": {
            "switch_type": "Blue",
            "backlight": "RGB",
            "connectivity": "Wired USB"
        }
    },
    {
        "id": "P005",
        "name": "4K Monitor",
        "category": "Electronics",
        "price": 399.99,
        "stock": 20,
        "description": "27-inch 4K UHD monitor with HDR support",
        "specs": {
            "resolution": "3840x2160",
            "size": "27 inches",
            "refresh_rate": "60Hz"
        }
    }
]

print(f"Loaded {len(PRODUCT_DATABASE)} products into the database")

## 3. Define the Agent Class

The agent will process queries and interact with the dataset intelligently.

In [ ]:
@dataclass
class QueryResult:
    """Structure to hold query results"""
    success: bool
    data: Any
    message: str
    query_type: str
    timestamp: str


class ProductAgent:
    """Simple agentic workflow for product queries"""
    
    def __init__(self, database: List[Dict]):
        self.database = database
        self.query_history = []
    
    def process_query(self, query: str) -> QueryResult:
        """
        Main method to process user queries.
        Determines query type and routes to appropriate handler.
        """
        query_lower = query.lower()
        timestamp = datetime.now().isoformat()
        
        # Log the query
        self.query_history.append({"query": query, "timestamp": timestamp})
        
        # Determine query type and route to handler
        if "price" in query_lower and "of" in query_lower:
            result = self._get_price_by_name(query)
        elif "stock" in query_lower or "available" in query_lower:
            result = self._check_stock(query)
        elif "category" in query_lower:
            result = self._get_by_category(query)
        elif "all products" in query_lower or "list" in query_lower:
            result = self._list_all_products()
        elif "specs" in query_lower or "specifications" in query_lower:
            result = self._get_specs(query)
        elif "search" in query_lower or "find" in query_lower:
            result = self._search_products(query)
        else:
            result = QueryResult(
                success=False,
                data=None,
                message="Query type not recognized. Try asking about price, stock, category, or specs.",
                query_type="unknown",
                timestamp=timestamp
            )
        
        return result
    
    def _get_price_by_name(self, query: str) -> QueryResult:
        """Get price of a product by name"""
        for product in self.database:
            if product["name"].lower() in query.lower():
                return QueryResult(
                    success=True,
                    data={"name": product["name"], "price": product["price"]},
                    message=f"The price of {product['name']} is ${product['price']}",
                    query_type="price_lookup",
                    timestamp=datetime.now().isoformat()
                )
        
        return QueryResult(
            success=False,
            data=None,
            message="Product not found",
            query_type="price_lookup",
            timestamp=datetime.now().isoformat()
        )
    
    def _check_stock(self, query: str) -> QueryResult:
        """Check stock availability"""
        for product in self.database:
            if product["name"].lower() in query.lower():
                in_stock = product["stock"] > 0
                return QueryResult(
                    success=True,
                    data={
                        "name": product["name"],
                        "stock": product["stock"],
                        "available": in_stock
                    },
                    message=f"{product['name']}: {'In stock' if in_stock else 'Out of stock'} ({product['stock']} units)",
                    query_type="stock_check",
                    timestamp=datetime.now().isoformat()
                )
        
        return QueryResult(
            success=False,
            data=None,
            message="Product not found",
            query_type="stock_check",
            timestamp=datetime.now().isoformat()
        )
    
    def _get_by_category(self, query: str) -> QueryResult:
        """Get all products in a category"""
        # Extract category from query
        categories = set(p["category"] for p in self.database)
        category_found = None
        
        for cat in categories:
            if cat.lower() in query.lower():
                category_found = cat
                break
        
        if category_found:
            products = [p for p in self.database if p["category"] == category_found]
            return QueryResult(
                success=True,
                data=products,
                message=f"Found {len(products)} products in {category_found}",
                query_type="category_search",
                timestamp=datetime.now().isoformat()
            )
        
        return QueryResult(
            success=False,
            data=None,
            message=f"Category not found. Available categories: {', '.join(categories)}",
            query_type="category_search",
            timestamp=datetime.now().isoformat()
        )
    
    def _list_all_products(self) -> QueryResult:
        """List all products"""
        return QueryResult(
            success=True,
            data=self.database,
            message=f"Found {len(self.database)} products",
            query_type="list_all",
            timestamp=datetime.now().isoformat()
        )
    
    def _get_specs(self, query: str) -> QueryResult:
        """Get specifications of a product"""
        for product in self.database:
            if product["name"].lower() in query.lower():
                return QueryResult(
                    success=True,
                    data={
                        "name": product["name"],
                        "specs": product["specs"],
                        "description": product["description"]
                    },
                    message=f"Specifications for {product['name']}",
                    query_type="specs_lookup",
                    timestamp=datetime.now().isoformat()
                )
        
        return QueryResult(
            success=False,
            data=None,
            message="Product not found",
            query_type="specs_lookup",
            timestamp=datetime.now().isoformat()
        )
    
    def _search_products(self, query: str) -> QueryResult:
        """Search products by keyword"""
        # Extract search terms (words after 'search' or 'find')
        query_lower = query.lower()
        search_terms = query_lower.split()
        
        results = []
        for product in self.database:
            # Search in name, description, and category
            searchable_text = f"{product['name']} {product['description']} {product['category']}".lower()
            
            if any(term in searchable_text for term in search_terms if len(term) > 2):
                results.append(product)
        
        if results:
            return QueryResult(
                success=True,
                data=results,
                message=f"Found {len(results)} matching products",
                query_type="search",
                timestamp=datetime.now().isoformat()
            )
        
        return QueryResult(
            success=False,
            data=None,
            message="No products found matching your search",
            query_type="search",
            timestamp=datetime.now().isoformat()
        )
    
    def get_query_history(self) -> List[Dict]:
        """Return query history"""
        return self.query_history


print("ProductAgent class defined successfully!")

## 4. Initialize the Agent

In [ ]:
# Create an instance of the agent
agent = ProductAgent(PRODUCT_DATABASE)
print("Agent initialized and ready to process queries!")

## 5. Test the Agentic Workflow

Let's test various types of queries to demonstrate the agent's capabilities.

In [ ]:
def display_result(result: QueryResult):
    """Helper function to display query results"""
    print("=" * 60)
    print(f"Query Type: {result.query_type}")
    print(f"Success: {result.success}")
    print(f"Message: {result.message}")
    print(f"Timestamp: {result.timestamp}")
    if result.data:
        print(f"\nData:")
        print(json.dumps(result.data, indent=2))
    print("=" * 60)
    print()

### Test 1: Price Lookup

In [ ]:
result = agent.process_query("What is the price of Laptop Pro 15?")
display_result(result)

### Test 2: Stock Check

In [ ]:
result = agent.process_query("Is USB-C Hub available in stock?")
display_result(result)

### Test 3: Category Search

In [ ]:
result = agent.process_query("Show me all products in the Electronics category")
display_result(result)

### Test 4: Product Specifications

In [ ]:
result = agent.process_query("What are the specs of the Mechanical Keyboard?")
display_result(result)

### Test 5: List All Products

In [ ]:
result = agent.process_query("List all products")
display_result(result)

### Test 6: Search Products

In [ ]:
result = agent.process_query("Search for wireless products")
display_result(result)

### Test 7: Multiple Queries

In [ ]:
queries = [
    "What is the price of 4K Monitor?",
    "Is Wireless Mouse available?",
    "Show me Accessories category",
    "Find keyboard"
]

for query in queries:
    print(f"\n>>> Query: {query}")
    result = agent.process_query(query)
    print(f"Response: {result.message}")
    print("-" * 60)

## 6. View Query History

In [ ]:
history = agent.get_query_history()
print(f"Total queries processed: {len(history)}\n")
print("Query History:")
for i, entry in enumerate(history, 1):
    print(f"{i}. {entry['query']} (at {entry['timestamp']})")

## 7. Interactive Query Interface

You can use this cell to test your own queries interactively.

In [ ]:
# Try your own query here!
my_query = "What is the price of Wireless Mouse?"

result = agent.process_query(my_query)
display_result(result)

## Summary

This notebook demonstrates a simple agentic workflow with the following features:

1. **Data Management**: Pre-defined product database
2. **Query Processing**: Intelligent routing based on query intent
3. **Multiple Query Types**:
   - Price lookups
   - Stock checks
   - Category searches
   - Specification queries
   - Full-text search
   - List all products
4. **History Tracking**: Maintains a log of all queries
5. **Structured Responses**: Returns data in a consistent format

### Possible Extensions:
- Add natural language processing for better query understanding
- Implement fuzzy matching for product names
- Add filtering and sorting capabilities
- Connect to external APIs or databases
- Implement conversation memory for context-aware responses
- Add analytics and insights generation